# Systolic-Array DNN Accelerator Simulator — Results & Validation
### Module scope: stationary schemes (OS/WS/IS) · memory layouts (Row/Column/Channel-major) · casting schemes (Multicast/Unicast/Hybrid) · analytical configuration-chooser

This notebook loads **real result files** produced by this project's Verilator + cocotb RTL verification suite and its Python analytical cost model, and presents them as tables and charts suitable for a live evaluation and for inclusion in a thesis.

**Every value below is labelled `measured (RTL)` or `model`.** `measured (RTL)` means the number came directly from a cycle-accurate Verilator simulation of the real hardware description, driven by a cocotb testbench, checked against a TensorFlow golden reference. `model` means the number came from the analytical Python cost formulas — computed instantly, with no hardware simulation. Nothing on this page is invented; every cell reads from a CSV/JSON file that ships in `data_bundle.zip`.

---
## How to run this notebook
1. Open **[colab.research.google.com](https://colab.research.google.com)** and upload this `.ipynb` file (File → Upload notebook), *or* open it directly from GitHub if the repo is pushed there.
2. Run the first code cell below. It will prompt you to **upload `data_bundle.zip`** — select the file from `results/thesis_notebook/data_bundle.zip` in this project.
3. Run all remaining cells top to bottom (Runtime → Run all). No installs are needed — everything used (`pandas`, `matplotlib`, `numpy`) is preinstalled in Colab.
4. For the thesis: right-click any chart → *Save image as*, or select a table cell's output and copy it — `pandas` tables also export directly with `df.to_latex()` if your thesis is in LaTeX (shown at the end).

In [ ]:
# ------------------------------------------------------------------
# Setup — upload data_bundle.zip when prompted (skip prompt if files
# are already present, e.g. when re-running locally with Jupyter).
# ------------------------------------------------------------------
import os, zipfile, sys
from pathlib import Path

BUNDLE_DIR = Path('data_bundle')
if not BUNDLE_DIR.exists():
    try:
        from google.colab import files
        print('Please select data_bundle.zip (from results/thesis_notebook/ in the project repo):')
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
        with zipfile.ZipFile(zip_name) as z:
            z.extractall('.')
    except ImportError:
        # Not running in Colab -- look for a local copy next to this notebook.
        local_zip = Path('data_bundle.zip')
        assert local_zip.exists(), 'Place data_bundle.zip next to this notebook, or run in Colab.'
        with zipfile.ZipFile(local_zip) as z:
            z.extractall('.')

assert BUNDLE_DIR.exists(), 'data_bundle/ not found after extraction.'
print('Data bundle ready:', sorted(p.name for p in BUNDLE_DIR.iterdir()))


In [ ]:
import json, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_colwidth', 80)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.grid': True,
                      'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})

# Consistent colour roles used throughout this notebook and this project's
# other results pages: teal = measured on real RTL, violet = analytical model.
MEASURED = '#0C7A73'
MODEL    = '#6B4FA0'
DF_COLOR = {'OS': '#2a78d6', 'IS': '#eb6834', 'WS': '#1baf7a'}

GC_RAW = BUNDLE_DIR / 'golden_check' / 'raw'
GC_FIG = BUNDLE_DIR / 'golden_check' / 'figures'
CHOOSER = BUNDLE_DIR / 'chooser'
EDGE_CLOUD = BUNDLE_DIR / 'edge_cloud_sample'

def source_tag(text, kind='measured'):
    """Small coloured badge so every printed table states its own provenance."""
    color = MEASURED if kind == 'measured' else MODEL
    label = 'MEASURED (RTL)' if kind == 'measured' else 'MODEL (analytical)'
    print(f'\033[1m[{label}]\033[0m {text}')


---
## 0 · What this module covers

This notebook documents the **stationary-scheme / memory-layout / casting-scheme** hardware knobs and the **analytical configuration-chooser** built on top of them.

| Knob | Options | What it controls |
|---|---|---|
| **Stationary scheme (dataflow)** | Output-Stationary (OS), Weight-Stationary (WS), Input-Stationary (IS) | Which operand stays resident in the PE array while the others stream through |
| **Memory layout** | Row-major, Column-major, Channel-major | The order tensors are laid out in off-chip DRAM, which changes how well consecutive fetches coalesce into AXI bursts |
| **Casting scheme** | Multicast, Unicast, Hybrid | How a value shared by multiple PEs is fetched off-chip: once (multicast), once per consuming PE (unicast), or split by operand (hybrid) |
| **Configuration chooser** | — | An analytical tool that scores all 3×3×3 = 27 combinations for a given workload, array size, and memory budget, and ranks them by a chosen goal (off-chip traffic / latency / energy / a weighted mix) — without running RTL per query |

The sections below prove, with real numbers: **(1)** the RTL implementing these knobs is functionally correct against a TensorFlow golden reference, **(2)** the traffic each knob produces is measured and understood, and **(3)** the chooser's analytical model, built on top of that measured behaviour, makes decisions that agree with it.

---
## 1 · Simulator correctness vs. the TensorFlow golden reference

Every RTL configuration below was run through the **same procedure**: Verilator compiles the SystemVerilog design into a cycle-accurate simulator; a cocotb Python testbench drives it with a real DNN layer's weights and inputs; the output is compared element-by-element against an independently computed TensorFlow result (the *golden reference*); a run passes if every element is within **5% of the layer's largest value** (a tolerance sized to absorb expected fixed-point quantization, not to hide real errors).

In [ ]:
f2 = pd.read_csv(GC_FIG / 'f2_correctness_margin.csv')
f2 = f2.sort_values('max_rel_err_pct', ascending=False).reset_index(drop=True)
TOL_PCT = 5.0
f2['tolerance_pct'] = TOL_PCT
f2['margin_x'] = TOL_PCT / f2['max_rel_err_pct']
f2['verdict'] = np.where(f2['max_rel_err_pct'] <= TOL_PCT, 'PASS', 'FAIL')

source_tag(f'{len(f2)} RTL configurations, correctness margin table', 'measured')
display_cols = ['config', 'family', 'max_rel_err_pct', 'tolerance_pct', 'margin_x', 'verdict']
table = f2[display_cols].copy()
table['max_rel_err_pct'] = table['max_rel_err_pct'].round(4)
table['margin_x'] = table['margin_x'].round(0).astype(int).astype(str) + 'x'
table


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = f2['family'].map(lambda f: DF_COLOR.get(f, '#9a9690'))
y = np.arange(len(f2))
ax.barh(y, f2['max_rel_err_pct'], color=colors, height=0.62)
ax.axvline(TOL_PCT, color='#d03b3b', linestyle='--', linewidth=1.5, label=f'{TOL_PCT:.0f}% tolerance (fail threshold)')
ax.set_yticks(y)
ax.set_yticklabels(f2['config'], fontsize=7)
ax.set_xlabel('Max relative error, % of full scale  (measured RTL vs. TensorFlow golden)')
ax.set_xlim(0, TOL_PCT * 1.15)
ax.set_title(f'RTL correctness margin — {len(f2)} configurations, all PASS, worst case {f2["max_rel_err_pct"].max():.3f}% (100x inside tolerance)')
handles = [plt.Rectangle((0,0),1,1, color=c) for c in DF_COLOR.values()] + [plt.Line2D([0],[0], color='#d03b3b', linestyle='--')]
ax.legend(handles, list(DF_COLOR.keys()) + ['5% tolerance'], loc='lower right', fontsize=8)
fig.tight_layout()
plt.show()
print(f'\nWorst-case error is {TOL_PCT / f2["max_rel_err_pct"].max():.0f}x inside the tolerance band -- '
      f'the residual error is consistent with pure fixed-point quantization, not a functional bug.')


---
## 2 · Off-chip traffic: memory layout and casting scheme

This is the direct evidence for the **layout** and **casting** knobs: how many AXI read requests, how many beats (data chunks), and how many cycles each setting produces, measured on the real AXI port during an RTL run.

In [ ]:
f4 = pd.read_csv(GC_FIG / 'f4_data_delivery_traffic.csv')
source_tag('Off-chip traffic by layout and by casting scheme', 'measured')
f4


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

lay = f4[f4['axis'] == 'layout']
ax = axes[0]
x = np.arange(len(lay))
ax.bar(x - 0.18, lay['axi_ar_requests'], width=0.36, color=MEASURED, label='AR requests')
ax2 = ax.twinx()
ax2.bar(x + 0.18, lay['total_cycles'], width=0.36, color='#c9c4b8', label='Total cycles')
ax.set_xticks(x); ax.set_xticklabels(lay['setting'], rotation=15)
ax.set_ylabel('AXI AR requests', color=MEASURED)
ax2.set_ylabel('Total cycles', color='#6b675c')
ax.set_title(f'Layout axis -- tiny L0, {int(lay["axi_beats"].iloc[0])} beats moved in every case\n(identical data volume; only request count / cycles differ)')

cast = f4[f4['axis'] == 'casting']
cast_tiny = cast[cast['layer'] == 'tiny L0']
ax = axes[1]
x = np.arange(len(cast_tiny))
ax.bar(x, cast_tiny['axi_beats'], color=[MEASURED, '#c9903a', '#b0463f'])
ax.set_xticks(x); ax.set_xticklabels(cast_tiny['setting'])
ax.set_ylabel('AXI beats (off-chip data volume)')
ax.set_title('Casting axis -- tiny L0, whole layer\n(same array, same layout -- volume changes by up to 11x)')
for i, v in enumerate(cast_tiny['axi_beats']):
    ax.text(i, v, f'{int(v):,}', ha='center', va='bottom', fontsize=9)

fig.tight_layout()
plt.show()


### Proof that the analytical formula reproduces this measured traffic exactly

The chooser's off-chip traffic model isn't a rough estimate — its casting-traffic formula was checked against every measured casting run this project recorded, and matches to the last beat.

In [ ]:
anchors = pd.read_csv(CHOOSER / 'eval_anchor_beats.csv')
anchors = anchors[anchors['anchor'].str.contains('beats')]
source_tag('Analytical-formula prediction vs. measured RTL beat count', 'model')
anchors_display = anchors[['anchor', 'model_value', 'measured_value', 'exact_match']].rename(
    columns={'model_value': 'model (formula)', 'measured_value': 'measured (RTL)', 'exact_match': 'exact match'})
anchors_display


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(anchors))
ax.bar(x - 0.19, anchors['model_value'], width=0.38, color=MODEL, label='model (formula)')
ax.bar(x + 0.19, anchors['measured_value'], width=0.38, color=MEASURED, label='measured (RTL)')
ax.set_xticks(x)
ax.set_xticklabels([a.replace(' beats', '').replace('(8x8)', '\n(8x8)') for a in anchors['anchor']], fontsize=7.5)
ax.set_ylabel('AXI beats')
n_match = int(anchors['exact_match'].sum())
ax.set_title(f'Model vs. measured: {n_match}/{len(anchors)} exact matches (bars are indistinguishable where they agree)')
ax.legend()
fig.tight_layout()
plt.show()


---
## 3 · Memory management: STAMP vs. PAGED, banked scratchpad

In [ ]:
f5 = pd.read_csv(GC_FIG / 'f5_memory_management.csv')
source_tag('STAMP vs PAGED off-chip bytes, and bank-conflict sweep', 'measured (mostly) -- see source column')
f5


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bytes_rows = f5[f5['metric'].str.contains('off-chip bytes')]
ax = axes[0]
layers = bytes_rows['layer'].unique()
x = np.arange(len(layers)); width = 0.35
stamp_vals = [bytes_rows[(bytes_rows['layer']==l) & (bytes_rows['metric'].str.contains('STAMP'))]['value'].iloc[0] for l in layers]
paged_vals = [bytes_rows[(bytes_rows['layer']==l) & (bytes_rows['metric'].str.contains('PAGED'))]['value'].iloc[0] for l in layers]
ax.bar(x - width/2, stamp_vals, width, color=MEASURED, label='STAMP (measured)')
ax.bar(x + width/2, paged_vals, width, color='#c9903a', label='PAGED (derived from measured page faults)')
ax.set_xticks(x); ax.set_xticklabels(layers)
ax.set_ylabel('Off-chip bytes')
ax.set_title('STAMP fetches less off-chip data than the PAGED baseline')
ax.legend()

bank_rows = f5[f5['metric'].str.contains('bank conflicts')]
ax = axes[1]
banks = [int(m.split('@ ')[1].split(' ')[0]) for m in bank_rows['metric']]
ax.plot(banks, bank_rows['value'], marker='o', color=MEASURED, linewidth=2)
ax.set_xscale('log', base=2)
ax.set_xticks(banks); ax.set_xticklabels(banks)
ax.set_xlabel('Number of scratchpad banks')
ax.set_ylabel('Bank conflicts (measured, real STAMP traffic)')
ax.set_title('Bank conflicts fall to zero as bank count grows\n(interleaved-banking arbitration behaving as designed)')

fig.tight_layout()
plt.show()


---
## 4 · The analytical configuration-chooser: is it correct?

The chooser never runs RTL — it scores all 27 (dataflow × layout × casting) combinations with the formulas validated above, in milliseconds. Its own correctness was evaluated four ways.

In [ ]:
acc = pd.read_csv(CHOOSER / 'eval_decision_accuracy.csv')
source_tag(f'(a) Decision accuracy: chooser top pick vs. independent exhaustive best, {len(acc)} (workload x goal) trials', 'model')
n_match = int(acc['match'].sum())
print(f'  Match: {n_match}/{len(acc)}  ({n_match/len(acc)*100:.0f}%)')
print(f'  Optimality gap when they differ: mean {acc["optimality_gap_pct"].mean():.4f}%, max {acc["optimality_gap_pct"].max():.4f}%  -- (b)')
acc.head(8)


In [ ]:
spd = pd.read_csv(CHOOSER / 'eval_speed.csv')
source_tag('(c) Speed: chooser (measured this run) vs. RTL lower bound (derived from recorded wall_seconds)', 'model + measured, mixed -- see source column')
spd


In [ ]:
decisions = pd.read_csv(CHOOSER / 'eval_anchor_decisions.csv')
source_tag('(d) Anchor check: chooser rankings vs. measured RTL, on the two workloads with real hardware runs', 'model, checked against measured (RTL)')
decisions


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
axes_matched = {
    'casting': decisions['casting_axis_matches_measured'].mean() * 100,
    'dataflow': decisions['dataflow_axis_matches_measured'].mean() * 100,
}
names = list(axes_matched.keys()); vals = list(axes_matched.values())
bars = ax.bar(names, vals, color=[MEASURED, '#c9903a'])
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+1.5, f'{v:.0f}%', ha='center', fontweight='bold')
ax.set_ylim(0, 110)
ax.set_ylabel('Agreement with measured RTL (%)')
ax.set_title('Chooser vs. measured hardware, by axis (tiny_cnn + mnist_cnn anchors)')
fig.tight_layout()
plt.show()
print('\nThe dataflow axis is the one open item: on --goal latency the chooser resolves the OS/IS gap at only')
print('~0.09%% while measured hardware differs by 11-14%% -- documented as an open limitation, not hidden.')


---
## 5 · The edge/cloud experiment suite — what it is, and what it is *not*

This project also contains a much larger sweep across 14 industry-scale reference DNNs (MobileNetV2, ResNet, BERT, DLRM, GPT-2, etc. — `results/edge/` and `results/cloud/`). **It is important to be precise about this suite in an evaluation: none of it is RTL-measured.** It is a pure analytical design-space exploration, generated by `scripts/run_full_eval.py`, using hand-authored cost formulas — the same family of formulas validated above, but applied to workloads that were never run through Verilator (they are far too large to simulate in RTL in reasonable time).

In [ ]:
exec_cyc = pd.read_csv(EDGE_CLOUD / 'edge_exp7_execution_cycles.csv')
source_tag('CAUTION -- despite the column name, this is NOT measured RTL data', 'model (synthetic)')
print("scripts/run_full_eval.py computes the 'rtl_actual' column as:")
print("    rtl_actual = python_estimated_cycles * a FIXED assumed multiplier (1.03-1.05, one per dataflow)")
print('That multiplier is an assumption written into the script, not something measured on this workload.')
exec_cyc.head(6)


In [ ]:
fpga = pd.read_csv(EDGE_CLOUD / 'edge_exp7_fpga_resources.csv')
source_tag('CAUTION -- FPGA utilisation numbers computed from a hand-authored formula, no synthesis tool run', 'model (synthetic)')
print('lut_util/dsp_util/bram_util/op_freq_norm come from a power-law formula (e.g. 22 * (array/8)**1.8),')
print('picked to look plausible -- no Vivado, no synthesis, no FPGA was ever involved in producing these.')
fpga


**How to describe this suite correctly in the evaluation, if asked:**
> "The edge/cloud suite is a design-space exploration tool: it applies the same cost-model formulas we validated against real RTL on tiny_cnn and mnist_cnn to a much larger set of reference DNNs, to show how trends generalize. It is explicitly labelled `model`, not `measured`, everywhere it appears in our reports — including the `exp7_hw_verification` file, whose `rtl_actual` and FPGA-utilisation columns are computed from assumed correction factors and a synthetic formula, not from a real hardware or FPGA run."

---
## 6 · Summary for the thesis / evaluation

| Question | Answer, in one line |
|---|---|
| Is the RTL functionally correct? | Yes — 26/26 measured configurations pass within 5% tolerance; worst case is 100x inside the band and consistent with fixed-point rounding, not a bug. |
| Is the off-chip traffic model validated? | Yes — the casting-traffic formula reproduces all 6 measured beat counts exactly; the analogous WS re-fetch formula (not shown above, see `CHOOSER_REPORT.md`) reproduces all 4 measured WS runs exactly. |
| Is the chooser's decision-making correct? | Its top pick matches exhaustive search 100% of the time (by construction, since it *is* exhaustive over 27 combos); on the two hardware-anchored workloads, it agrees with measured RTL on the casting and layout axes, and is honestly documented as under-resolving the dataflow-latency axis. |
| Are the edge/cloud numbers measured? | No — clearly labelled `model`, included to show the same formulas generalize, not as hardware evidence. |

### Exporting for a LaTeX thesis
Any table above can be exported directly, e.g.:

In [ ]:
# Example: export the correctness-margin table for the thesis.
table.to_csv('correctness_margin_export.csv', index=False)
print('Wrote correctness_margin_export.csv -- download it from the Colab file browser (folder icon, left side).')
try:
    # to_latex() needs the optional 'jinja2' package; not guaranteed present everywhere.
    latex_table = table.to_latex(index=False)
    print('\nLaTeX table (requires jinja2):\n')
    print(latex_table[:600], '...')
except ImportError:
    print("\n(LaTeX export needs 'jinja2' -- run `!pip install jinja2` in a cell above and re-run this one if you need it.)")
